# QuEL-3 capture delay の探索・確認

QuEL-3 の `capture_delay` をns単位（校正は32 ns刻み）で掃引し、パルス全体の中点をキャプチャ窓の中央に配置します。QuEL-3では `capture_delay_word` は使用しません。

各測定後に遅延設定を復元します。設定ファイルへの保存は `save_params()` を明示的に呼んだ場合だけ行います。ノートブックの実行環境に `.venv` を選び、このリポジトリのルートを作業ディレクトリにしてください。


## 1. 装置の準備

In [ ]:
import os
from pathlib import Path

import qubex as qb

from capture_delay_tools import CaptureDelayCalibrator

config_root = (
    Path(os.environ.get("QUBEX_CONFIG_ROOT", "./qubex-config"))
    .expanduser()
    .resolve()
)
system_id = "144Qv3-quel3"
exp = qb.Experiment(
    system_id=system_id,
    config_dir=config_root / system_id / "config",
    params_dir=config_root / system_id / "params",
    muxes=[6],  # 実験対象に合わせて変更
)


In [ ]:
exp.connect()

In [ ]:
exp.configure()

In [ ]:
for qubit in exp.qubit_labels:
    exp.check_waveform(qubit,readout_amplitude =1.0)

## 2. ns単位の掃引

QuEL-3では `capture_delay` にns値を直接指定します。値は読み出しサンプリング周期の0.8 nsの倍数である必要があります。まず32 ns刻みで粗く掃引し、必要なら候補周辺を0.8 ns刻みで調べます。

|IQ|ヒートマップの横軸はcapture delay（ns）、縦軸はキャプチャ内時刻（ns）、色は振幅です。平均回数は掃引・再検証・最終確認で共通です。


### キャプチャ遅延確認用の固定周波数

共鳴周波数の探索前に実行するため、`readout_frequencies_ghz` に MUX ごとの絶対周波数（GHz）を指定します。装置・配線・フィルタの通過帯域に合わせて設定してください。同じ条件の MUX には共通値を使えます。未指定の MUX は測定前にエラーにします。`check_waveform` を1量子ビットずつ実行し、`readout_amplitude=1.0` を指定します。周波数の変更・復元は MUX ごとの掃引全体で1回にまとめ、掃引・再検証・最終確認で同じ周波数を使います。測定後は設定を戻し、周波数の YAML は変更しません。

In [ ]:
# 接続先の帯域に合わせて入力してください。例: {0: 9.0}（9 GHz）
# 9 GHz は記法の例であり、装置共通の推奨値ではありません。
readout_frequencies_ghz = {index: 7 for index in range(16)}
readout_frequencies_ghz

In [ ]:
system = exp.system_manager.experiment_system
mux_indices = sorted({system.get_mux_by_qubit(q).index for q in exp.qubit_labels})
if not mux_indices:
    raise ValueError("Experimentのmuxes/qubitsと設定を確認してください。対象が空です。")

# 粗い掃引: 0〜2304 nsを32 ns刻み（すべて0.8 nsグリッド上）
delay_sweep_ns = {mux: range(0, 2305, 32) for mux in mux_indices}

# 候補周辺を32 ns刻みで調べる例:
# delay_sweep_ns = {mux: range(640, 993, 32) for mux in mux_indices}

calibrator = CaptureDelayCalibrator(
    exp,
    readout_frequencies_ghz=readout_frequencies_ghz,
    n_shots=1024,
    snr_threshold=5.0,
    smoothing=5,
)
# QuEL-3では各MUXの対象量子ビットを1つずつcheck_waveformで測定します。
# readout_amplitude=1.0はcalibrator内部で指定します（再検証・最終確認も共通）。
display(calibrator.sweep(delay_sweep_ns))
calibrator.plot_sweep()


## 3. エッジ解析と候補選択

赤は立ち上がり、橙は立ち下がり、破線は窓中央です。全対象で整合する観測を使い、前後32 ns以上の余白を確保できる候補を32 nsグリッド上で選びます。候補が得られない場合は `selection` の `reason` を確認し、掃引範囲を広げてください。


In [ ]:
# 必要なら解析窓を指定: calibrator.analyze(windows={"Q00": (0, 500)})
_ = calibrator.analyze()
display(calibrator.select())
calibrator.plot_analysis()


## 4. 候補位置で再測定

選択したcapture delay（ns）を一時適用し、両端を再検出します。`error_ns` は実測中点とキャプチャ窓中央との差です。


In [ ]:
display(calibrator.selected_delays)
display(calibrator.verify())
calibrator.plot_verification()

## 5. 最終capture delayの決定

全対象の両端が再測定で検出できた場合に限り、前後32 ns以上の余白を確保しつつ、窓中央に最も近い値を32 nsグリッド上で選びます。


In [ ]:
display(calibrator.finalize())
display({"capture_delay_ns": calibrator.final_delays})


## 6. 最終波形の確認

最終capture delayを適用して再測定します。I/Q波形全体がキャプチャ窓に収まり、前後の余白が十分であることを確認してください。詳細は `analysis`、`verification`、`final_report` と各waveformの `total_delay_ns` に残ります。


In [ ]:
if not calibrator.final_delays and not calibrator.verification.empty:
    display(calibrator.finalize())

if calibrator.final_delays:
    display({"capture_delay_ns": calibrator.final_delays})
    calibrator.check_final()
else:
    display(calibrator.selection)
    display(calibrator.verification)
    display(calibrator.final_report)
    print("採用できる最終値がありません。reasonと掃引範囲を確認してください。")


## 7. 最終候補の保存（任意）

QuEL-3では `capture_delay.yaml` だけを `meta.unit: ns` で保存します。既存ファイルを更新する場合も、採用したMUX以外の値・メタデータ・コメントを保持します。最終波形を確認してから保存セルのコメントを外してください。


In [ ]:
# calibrator.save_params("/path/to/params")
# calibrator.save_params("/path/to/params", overwrite=True)


### 解析環境

`TimeOfFlightAnalysis` は、このリポジトリのPython 3.13環境にインストールした `sqe-analysis` を直接呼び出します。別プロセスや実行時の依存インストールは行いません。固定リビジョンは `pyproject.toml` と `uv.lock` を参照してください。


### 更新後の再実行

カーネルを再起動し、準備セルから新しい `calibrator` を作成してください。
